<a href="https://colab.research.google.com/github/alearecuest/anyoneai-exercises-sprint_3/blob/main/17_7_2_RAG_Agent_Toy_Example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG toy example and example

This notebook demonstrates a minimal Retrieval Augmented Generation (RAG) pipeline in Google Colab using:
- PDF documents downloaded with `gdown`,
- text extraction and chunking,
- embeddings via `sentence-transformers`,
- FAISS as the vector store,
- a generator model (`Qwen/Qwen2.5-0.5B-Instruct`) for answer generation,
- evaluation with the RAGAS metric from the `ragas` library.

## 1) Ensure GPU runtime and check device

Please set Runtime -> Change runtime type -> GPU, then run the next cell to confirm.

In [ ]:
# run to confirm GPU availability
import torch, sys
print("Python:", sys.version.splitlines()[0])
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device name:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Please enable a GPU under Runtime -> Change runtime type.")

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Torch version: 2.8.0+cu126
CUDA available: True
CUDA device name: Tesla T4


In [ ]:
# Install necessary packages.
# Note: Colab often already has many of these; pip will upgrade/install what's missing.
# We use faiss-cpu to avoid CUDA mismatch issues, and keep generator model moderate sized for Colab GPU.
!pip install -q ragas==0.3.5 sentence-transformers transformers[torch] faiss-cpu pdfplumber langchain accelerate rouge-score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.3/284.3 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

## 2) Download PDF files with gdown
If you have multiple PDFs, repeat or expand this cell with different filenames and ids.


In [ ]:
import os
os.makedirs("/content/data", exist_ok=True)

# replace with your ID(s)
FILE_ID = "1EbDhFJlmxKtzcEZTurqw6KbNW9wJMMDR"
OUT_PATH = "/content/data/docs.pdf"

# gdown is available via pip in Colab, but using the CLI is often simplest
!gdown --id {FILE_ID} -O {OUT_PATH} || echo "gdown failed; make sure file id is correct and file is publicly accessible"

print("Saved to", OUT_PATH)

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1EbDhFJlmxKtzcEZTurqw6KbNW9wJMMDR
To: /content/data/docs.pdf
100% 1.05M/1.05M [00:00<00:00, 90.4MB/s]
Saved to /content/data/docs.pdf


## 3) Extract text from the downloaded PDF(s) and create a simple list of documents.

This uses pdfplumber. Each page will be one text chunk initially; we will chunk more smartly later.

In [ ]:
import pdfplumber, glob
from pathlib import Path

pdf_paths = glob.glob("/content/data/*.pdf")
docs = []
for p in pdf_paths:
    with pdfplumber.open(p) as pdf:
        pages_text = [page.extract_text() or "" for page in pdf.pages]
    # join or keep per-page; keep per-page for now
    for i, txt in enumerate(pages_text):
        content = txt.strip()
        if content:
            docs.append({"source": Path(p).name, "page": i+1, "text": content})

print(f"Loaded {len(docs)} page-documents from {len(pdf_paths)} pdf(s).")
# show example
if docs:
    print("Example (first 500 chars):")
    print(docs[2]["text"][:500])

Loaded 12 page-documents from 1 pdf(s).
Example (first 500 chars):
Consensus Statement https://doi.org/10.1038/s41591-024-03425-5
Table 1 | Research design and LLM task categories for the modular TRIPOD-LLM guidelines
Task Definition Example
Research design
De novo LLM Building a new language model from scratch or substantially fine-tuning A study pretraining a new LLM on a hospital’s clinical
development existing base models to develop new functionalities or to adapt to new tasks. data, for example, ref. 56
LLM methods Quantitative or theoretical investigation


## 4) Chunk the text into smaller passages suitable for retrieval.
We use a naive sliding-window chunker with overlap.

In [ ]:
def chunk_text(text, chunk_size=500, overlap=100):
    tokens = text.split()
    chunks = []
    i = 0
    while i < len(tokens):
        chunk = " ".join(tokens[i:i+chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap
    return chunks

corpus = []
for doc in docs:
    chunks = chunk_text(doc["text"], chunk_size=400, overlap=50)
    for idx, c in enumerate(chunks):
        corpus.append({
            "id": f"{doc['source']}_p{doc['page']}_c{idx}",
            "source": doc["source"],
            "page": doc["page"],
            "chunk_index": idx,
            "text": c
        })

print("Total chunks:", len(corpus))
print("First chunk preview:\n", corpus[2]["text"][:400])

Total chunks: 36
First chunk preview:
 Consensus Statement https://doi.org/10.1038/s41591-024-03425-5 recommended by journals and is often included in journal instructions The new complexities introduced by LLMs include concerns to authors. TRIPOD has subsequently been updated to incorporate regarding hallucinations, omissions, reliability, explainability, repro- best practices for artificial intelligence (AI) due to the substantially 


## 5) Create embeddings (sentence-transformers) and index with FAISS.
Use an embedding model that is small but effective. The embedding computation will use GPU if torch + CUDA available.


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
import pickle

EMB_MODEL = "all-mpnet-base-v2"  # good balance of quality and speed
print("Loading embedding model:", EMB_MODEL)
embedder = SentenceTransformer(EMB_MODEL)

texts = [c["text"] for c in corpus]
ids = [c["id"] for c in corpus]

# compute embeddings (batching)
batch_size = 32
embs = embedder.encode(texts, batch_size=batch_size, show_progress_bar=True, convert_to_numpy=True)

# build FAISS index
dimension = embs.shape[1]
index = faiss.IndexFlatIP(dimension)  # inner product; we will normalize vectors
faiss.normalize_L2(embs)
index.add(embs)

# save index and metadata
os.makedirs("/content/index", exist_ok=True)
faiss.write_index(index, "/content/index/faiss.index")
with open("/content/index/metadata.pkl", "wb") as f:
    pickle.dump({"ids": ids, "corpus": corpus}, f)

print("FAISS index saved to /content/index/faiss.index")

Loading embedding model: all-mpnet-base-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

FAISS index saved to /content/index/faiss.index


## 6) Retriever helper that queries the FAISS index and returns top k passages.

In [ ]:
import faiss, pickle
from sentence_transformers import SentenceTransformer
import numpy as np

# load index and metadata
index = faiss.read_index("/content/index/faiss.index")
with open("/content/index/metadata.pkl","rb") as f:
    meta = pickle.load(f)
ids = meta["ids"]
corpus = meta["corpus"]

embedder = SentenceTransformer(EMB_MODEL)

def retrieve(query, top_k=5):
    q_emb = embedder.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    D, I = index.search(q_emb, top_k)
    results = []
    for score, idx in zip(D[0], I[0]):
        if idx < len(corpus):
            results.append({"score": float(score), "id": corpus[idx]["id"], "text": corpus[idx]["text"], "source": corpus[idx]["source"], "page": corpus[idx]["page"]})
    return results

# quick test
print(retrieve("Long-form question answering", top_k=1))

[{'score': 0.38150182366371155, 'id': 'docs.pdf_p2_c1', 'text': 'methods (eight items), open science practices (one item), sequence given the words that preceded it. Yet this foundational train- patient and public involvement (one item), results (three items) and ing has been shown to equip them with capabilities to perform a wide discussion (two items). These main items are further divided into 50 range of healthcare-related natural language processing (NLP) tasks subitems. Of these, 14 main items and 32 subitems are applicable to all from a single model. This adaptability is commonly achieved through research designs and LLM tasks. The remaining 5 main items and 18 subi- supervised fine-tuning or few-shot learning methods, which allow LLMs tems are specific to particular research designs or LLM task categories. to handle new tasks with minimal examples15,16. Chatbot solutions (for As discussed in the Methods, the TRIPOD-LLM statement introduces a example, ChatGPT) use LLMs as their f

## 7) Setup generator model and a RAG function that conditions the generator on the retrieved context.
We use transformers pipeline for text2text-generation with a moderate model that fits on Colab GPU.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

GEN_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
print("Loading generator model:", GEN_MODEL)
tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

# Create a text-generation pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=False,
    return_full_text=False
)

def rag_answer(query, top_k=4):
    # retrieve
    retrieved = retrieve(query, top_k=top_k)
    # combine context
    context = "\n\n".join([f"Source: {r['source']} (page {r['page']})\n{r['text']}" for r in retrieved])

    # Chat template for Qwen
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Use the provided context to answer the question. If the answer is not contained in the context, say 'I do not know'."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}
    ]

    # Apply chat template
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    out = generator(prompt)
    return {"query": query, "answer": out[0]["generated_text"].strip(), "retrieved": retrieved}

# quick example (replace with a question relevant to your PDF)
res = rag_answer("What does TRIPOD stand for?", top_k=3)
print("Answer:", res["answer"])

Loading generator model: Qwen/Qwen2.5-0.5B-Instruct


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Answer: TRIPOD stands for "The Living Systematic Review and Documentation."


## 8) Evaluation Samples

In [ ]:

# These samples were created from the uploaded TRIPOD-LLM document and are intended
# for use with the RAG evaluation pipeline. Each item contains 'query' and 'reference'.
eval_samples = [
    {
        "query": "What is TRIPOD-LLM?",
        "reference": "An extension of TRIPOD+AI providing a checklist for transparent reporting of studies using large language models in healthcare."
    },
    {
        "query": "How many main items and subitems are in the TRIPOD-LLM checklist?",
        "reference": "TRIPOD-LLM provides 19 main items and 50 subitems."
    },
    {
        "query": "What is the purpose of the TRIPOD-LLM interactive website?",
        "reference": "To present required questions based on research design(s) and task(s), allow easy completion and render a final PDF suitable for submission."
    },
    {
        "query": "Name three noteworthy changes introduced in TRIPOD-LLM (Box 1).",
        "reference": "A new dedicated checklist for LLMs; a living guideline (regular updates); and task-specific guidance tailored to LLM applications in healthcare."
    },
    {
        "query": "Which key task categories are listed for TRIPOD-LLM (examples)?",
        "reference": "Examples include long-form question answering, information retrieval, conversational agents (chatbots), documentation generation, summarization and simplification, machine translation and outcome forecasting."
    },
    {
        "query": "What does item 5a of the checklist ask authors to report?",
        "reference": "Describe the sources of data separately for training, tuning and/or evaluation datasets and the rationale for using these data."
    },
    {
        "query": "What metrics does item 7a recommend including for generative outputs?",
        "reference": "Metrics that capture consistency, relevance, accuracy and presence/type of errors compared to gold standards."
    },
    {
        "query": "Who are the primary intended users and beneficiaries of TRIPOD-LLM?",
        "reference": "Academic and industry researchers, journal editors and peer reviewers, and other stakeholders such as policymakers, funders, regulators, patients and study participants."
    },
    {
        "query": "What open-science items should be reported (brief)?",
        "reference": "Report source of funding and role of funders; conflicts of interest; protocol access; registration information; availability of study data and code."
    },
    {
        "query": "How should studies that include imaging (multimodal) report aspects differently?",
        "reference": "For vision-language or multimodal models, both text and image preprocessing and image acquisition details should be reported, with additional considerations possible in future versions."
    }
]

print(f"Prepared {len(eval_samples)} evaluation samples for RAG evaluation.")

Prepared 10 evaluation samples for RAG evaluation.


## 9) Run RAG over evaluation samples to produce generated answers and save context for Ragas evaluation.

In [ ]:

generated_results = []
for s in eval_samples:
    out = rag_answer(s["query"], top_k=4)
    generated_results.append({
        "query": s["query"],
        "generated": out["answer"],
        "reference": s["reference"],
        "retrieved": out["retrieved"]  # list of retrieved passages
    })

# show
for gr in generated_results:
    print("Q:", gr["query"])
    print("Gen:", gr["generated"][:300])
    print("Ref:", gr["reference"][:300])
    print("-"*60)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Q: What is TRIPOD-LLM?
Gen: TRIPOD-LLM is a tool for reviewing and evaluating machine learning (ML) models, particularly those used in natural language processing (NLP). It allows researchers to compare and evaluate various ML models using a standardized set of criteria. The tool is part of the TriPOD-LLM system, which aims to
Ref: An extension of TRIPOD+AI providing a checklist for transparent reporting of studies using large language models in healthcare.
------------------------------------------------------------
Q: How many main items and subitems are in the TRIPOD-LLM checklist?
Gen: The TRIPOD-LLM checklist has 59 main items and 50 subitems.
Ref: TRIPOD-LLM provides 19 main items and 50 subitems.
------------------------------------------------------------
Q: What is the purpose of the TRIPOD-LLM interactive website?
Gen: The purpose of the TRIPOD-LLM interactive website is to provide a platform for users to easily update and manage the content of the guidelines. This allows for

## Understanding the Evaluation Metrics

This section explains the meaning of each metric computed in the simplified evaluation step.

---

### **1. ROUGE-L**
**What it measures:**  
The overlap between the *generated answer* and the *reference (ground truth)* answer, based on the *longest common subsequence (LCS)* of words.

**Interpretation:**  
- High ROUGE-L → The generated answer textually matches the correct answer well.  
- Low ROUGE-L → The generated answer diverges from the reference (either incomplete or phrased differently).

**Use:**  
A general proxy for answer quality and textual similarity.

---

### **2. Context Precision**
**Definition:**  
The proportion of tokens in the *generated answer* that also appear in the *retrieved context passages*.

**Formula:**  

$$
\text{Context Precision} = \frac{\text{# tokens in generated answer also in retrieved text}}{\text{total # tokens in generated answer}}
$$

**Interpretation:**  
- High → The model primarily used content that appears in the retrieved passages (indicating grounded, faithful answers).  
- Low → The model introduced content not supported by the retrieved context (possible hallucination).

---

### **3. Context Recall**
**Definition:**  
The proportion of tokens in the *reference (ground truth)* answer that are present in the *retrieved context*.

**Formula:**  
$$
\text{Context Recall} = \frac{\text{# tokens in reference answer also in retrieved text}}{\text{total # tokens in reference answer}}
$$

**Interpretation:**  
- High → The retrieved passages contain most of the information needed for the correct answer (good retrieval coverage).  
- Low → Important information from the reference was missing in the retrieved content (poor retrieval).

---

### **4. Context Coverage**
**Definition:**  
In this simplified framework, *coverage* is equivalent to **context recall** and is included to highlight how much of the true answer is represented in the retrieved material.

---

### **Summary of Interpretation**

| Metric | Measures | High Value Means | Low Value Means |
|---------|-----------|------------------|-----------------|
| **ROUGE-L** | Overlap between generated and reference answers | Good textual alignment | Divergent or incomplete answer |
| **Context Precision** | Grounding of generated text in retrieved context | Faithful, low hallucination | Unsupported or hallucinated text |
| **Context Recall** | Retrieval coverage of correct information | Relevant passages retrieved | Missing key information |
| **Context Coverage** | Portion of true answer covered by retrieval | Good evidence coverage | Incomplete retrieval |

---

### **Typical Ranges (Rule of Thumb)**
- **ROUGE-L > 0.5** → Acceptable content similarity.  
- **Precision > 0.7** → Generation mostly grounded in retrieved text.  
- **Recall > 0.5** → Retrieval covered most relevant content.

These thresholds are heuristic. They are most useful for **comparing models or retrieval strategies** within the same experimental setup.

In [ ]:
# Simple deterministic RAG evaluation (ROUGE-L + context overlap metrics)
import json, re, statistics
from rouge_score import rouge_scorer
from pprint import pprint

# Helper: simple tokeniser (word characters)
def tokenize(text):
    if not text:
        return []
    return re.findall(r"\w+", text.lower())

# Compute ROUGE-L using rouge_score library
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

# Use 'generated_results' from the earlier cells. Each entry expected to have:
# {'query': ..., 'generated': ..., 'reference': ..., 'retrieved': [ { 'id':..., 'text':... }, ... ] }
if 'generated_results' not in globals():
    raise RuntimeError("generated_results not found. Run the RAG generation cell first.")

per_sample = []
for entry in generated_results:
    query = entry.get("query", "")
    generated = (entry.get("generated") or "").strip()
    reference = (entry.get("reference") or "").strip()

    # ROUGE-L score
    rouge_scores = scorer.score(reference, generated)
    rouge_l_f = rouge_scores['rougeL'].fmeasure  # F1-like score for ROUGE-L

    # Build a single string of all retrieved context text
    retrieved_texts = [r.get("text","") for r in entry.get("retrieved", [])]
    combined_context = " ".join(retrieved_texts)

    # Token counts
    gen_tokens = tokenize(generated)
    ref_tokens = tokenize(reference)
    ctx_tokens = set(tokenize(combined_context))

    # Context precision: fraction of generated tokens also present in contexts
    ctx_prec = 0.0
    if gen_tokens:
        matched_gen = sum(1 for t in gen_tokens if t in ctx_tokens)
        ctx_prec = matched_gen / len(gen_tokens)

    # Context recall: fraction of reference tokens present in contexts
    ctx_rec = 0.0
    if ref_tokens:
        matched_ref = sum(1 for t in ref_tokens if t in ctx_tokens)
        ctx_rec = matched_ref / len(ref_tokens)

    # Context coverage is same as ctx_rec but provided separately for clarity
    ctx_cov = ctx_rec

    per_sample.append({
        "query": query,
        "generated": generated,
        "reference": reference,
        "rouge_l_f": rouge_l_f,
        "context_precision": ctx_prec,
        "context_recall": ctx_rec,
        "context_coverage": ctx_cov,
        "retrieved_ids": [r.get("id") for r in entry.get("retrieved", [])]
    })

# Aggregate statistics (means and standard deviations)
def mean_std(values):
    if not values:
        return {"mean": None, "std": None}
    return {"mean": statistics.mean(values), "std": statistics.pstdev(values)}

rouge_vals = [s["rouge_l_f"] for s in per_sample]
prec_vals = [s["context_precision"] for s in per_sample]
rec_vals = [s["context_recall"] for s in per_sample]
cov_vals = [s["context_coverage"] for s in per_sample]

summary = {
    "n_samples": len(per_sample),
    "rouge_l": mean_std(rouge_vals),
    "context_precision": mean_std(prec_vals),
    "context_recall": mean_std(rec_vals),
    "context_coverage": mean_std(cov_vals)
}

results = {"summary": summary, "per_sample": per_sample}

# Save results
out_path = "/content/simple_rag_eval.json"
with open(out_path, "w") as f:
    json.dump(results, f, indent=2)

print("Saved simple RAG evaluation to", out_path)
print("\nSummary:")
pprint(summary)
print("\nPer-sample preview (first 3):")
pprint(per_sample[:3])

Saved simple RAG evaluation to /content/simple_rag_eval.json

Summary:
{'context_coverage': {'mean': 0.7268233627547128, 'std': 0.19733168191287018},
 'context_precision': {'mean': 0.6872222873521877, 'std': 0.1693338251566479},
 'context_recall': {'mean': 0.7268233627547128, 'std': 0.19733168191287018},
 'n_samples': 10,
 'rouge_l': {'mean': 0.20016137772045137, 'std': 0.1895898768360351}}

Per-sample preview (first 3):
[{'context_coverage': 0.8421052631578947,
  'context_precision': 0.4745762711864407,
  'context_recall': 0.8421052631578947,
  'generated': 'TRIPOD-LLM is a tool for reviewing and evaluating machine '
               'learning (ML) models, particularly those used in natural '
               'language processing (NLP). It allows researchers to compare '
               'and evaluate various ML models using a standardized set of '
               'criteria. The tool is part of the TriPOD-LLM system, which '
               'aims to standardize the process of evaluating NLP m

## This is not very good... Can you improve it?
Think about things like:
- were the chunks well created?
- are the embeddings doing a good job?
- is the retrieve working well?
- is the indexing fine?
- is the generator model that we picked the best one?
- do you understand what the metrics are doing?
- could the eval be improved? Check this out: https://docs.ragas.io/en/stable/